# Stage 08c: Corrected Segmentation Strategy

**Basis:** Stage 06c2 corrected official model (HistGradientBoostingClassifier, AUC≈0.8629) + Stage 07c TRUE SHAP

**Scope:**
- Regenerate compact official prediction scores from the Stage 06c2 official model
- Create risk bands from `churn_risk_score`
- Create non-exclusive segment flags and hierarchical final segments
- Compare with old 08b segments
- Write all required 08c outputs

**Not in scope:** model training, Optuna, SHAP, business simulation, model tuning

**Important:** `is_repurchase_label` is NOT used to define segments — only to evaluate churn rate after assignment.

In [ ]:
import sys
from pathlib import Path

# Add notebooks directory to path
nb_dir = Path.cwd()
if nb_dir.name != 'notebooks':
    # If running from project root, find notebooks dir
    nb_dir = Path('park.ingyeom/notebooks')
sys.path.insert(0, str(nb_dir))

print(f'Notebook dir: {nb_dir}')
print(f'Python: {sys.version}')

In [ ]:
# Run the implementation
from importlib import import_module, reload

try:
    impl = reload(import_module('08c_v2_corrected_segmentation_strategy_impl'))
except Exception:
    import importlib.util
    spec = importlib.util.spec_from_file_location(
        'impl',
        nb_dir / '08c_v2_corrected_segmentation_strategy_impl.py'
    )
    impl = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(impl)

result = impl.main()

In [ ]:
# Display key results
import pandas as pd
from pathlib import Path

PROJECT_ROOT = impl.PROJECT_ROOT
TABLE_DIR = impl.TABLE_DIR
DATA_DIR = impl.DATA_DIR

print(f'=== AUC Check ===')
print(f'Reconstructed AUC: {result["auc_test"]:.6f}')
print(f'Expected AUC:      {impl.EXPECTED_AUC:.6f}')
print(f'Diff:              {abs(result["auc_test"] - impl.EXPECTED_AUC):.10f}')
print()
print(f'=== Holdout N ===' )
print(f'n = {result["holdout_n"]}')

In [ ]:
# Risk band summary
band_df = pd.read_csv(TABLE_DIR / '08c_risk_band_summary_holdout.csv')
print('=== Risk Band Summary (Holdout) ===')
display(band_df[['risk_band', 'n', 'share', 'churn_rate', 'lift_vs_overall_churn_rate',
                  'captured_churners', 'churner_capture_rate', 'avg_churn_risk_score']])

In [ ]:
# Hierarchical segment summary
seg_df = pd.read_csv(TABLE_DIR / '08c_hierarchical_segment_summary_holdout.csv')
print('=== Hierarchical Segment Summary (Holdout) ===')
display(seg_df[['final_segment_key', 'n', 'share', 'churn_rate', 'lift_vs_overall_churn_rate',
                 'captured_churners', 'churner_capture_rate', 'avg_churn_risk_score', 'stability_note']])

In [ ]:
# Non-exclusive flag summary
flag_df = pd.read_csv(TABLE_DIR / '08c_nonexclusive_segment_flag_summary.csv')
print('=== Non-Exclusive Segment Flag Summary (Holdout) ===')
display(flag_df)

In [ ]:
# Thresholds
thr_df = pd.read_csv(TABLE_DIR / '08c_segment_thresholds.csv')
print('=== Segment Thresholds ===')
display(thr_df)

In [ ]:
# Old 08b vs new 08c comparison
cmp_df = pd.read_csv(TABLE_DIR / '08c_old08b_vs_new08c_comparison.csv')
print('=== Old 08b vs New 08c Comparison ===')
display(cmp_df[['old_08b_segment_ko', 'old_08b_churn_rate', 'new_08c_segment_ko', 'status', 'reason']])

In [ ]:
# Segment overlap matrix
overlap_df = pd.read_csv(TABLE_DIR / '08c_segment_overlap_matrix.csv')
print('=== Non-Exclusive Segment Overlap Matrix (Holdout) ===')
display(overlap_df)

In [ ]:
# Action recommendations
act_df = pd.read_csv(TABLE_DIR / '08c_segment_action_recommendations.csv')
print('=== Segment Action Recommendations ===')
display(act_df[['final_segment_key', 'n_holdout', 'churn_rate_holdout',
                 'recommended_action_ko', 'presentation_readiness', 'use_in_stage09c_simulation']])

In [ ]:
# Final checks
checks_df = pd.read_csv(TABLE_DIR / '08c_final_checks.csv')
print('=== Final Checks ===')
display(checks_df)

In [ ]:
# Display figures
from IPython.display import Image, display as ipy_display
FIGURE_DIR = impl.FIGURE_DIR

for fig_name in [
    '08c_risk_band_churn_rate_holdout.png',
    '08c_risk_band_lift_holdout.png',
    '08c_hierarchical_segment_size_and_churn.png',
    '08c_segment_action_map.png',
    '08c_segment_shap_evidence_heatmap.png',
    '08c_top_decile_churn_capture.png',
]:
    fig_path = FIGURE_DIR / fig_name
    if fig_path.exists():
        print(f'--- {fig_name} ---')
        ipy_display(Image(filename=str(fig_path), width=700))
    else:
        print(f'MISSING: {fig_name}')

In [ ]:
# Show business readiness findings
biz_df = pd.read_csv(TABLE_DIR / '08c_business_readiness_findings.csv')
print('=== Business Readiness Findings ===')
display(biz_df)

In [ ]:
# Show segmentation summary JSON
import json
summary = json.loads((DATA_DIR / '08c_segmentation_summary.json').read_text(encoding='utf-8'))
print('=== 08c Segmentation Summary JSON ===')
for k, v in summary.items():
    print(f'  {k}: {v}')

## Summary

Stage 08c corrected segmentation is complete.

- Official model: HistGradientBoostingClassifier (Stage 06c2, AUC=0.8629)
- Official XAI basis: Stage 07c TRUE SHAP
- 4 risk bands + 6 hierarchical segments created
- Old 08b comparison table created
- All required outputs written under `08c_` prefix
- **Do not proceed to Stage 09c from this notebook.**